# **[Tower Dataset](https://data.go.id/dataset/dataset/tower-telekomunikasi-50k)**

The dataset is taken from **Satu Data Indonesia**

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/lokasi_menara_telekomunikasi.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

### Data Information
Let's check the data types and non-null values for each column.

In [ ]:
# Get information about the DataFrame, including data types and non-null values
display(df.info())

### Descriptive Statistics
Now, let's look at the summary statistics for numerical columns.

In [ ]:
# Display descriptive statistics for numerical columns
display(df.describe())

### Missing Values
Finally, let's identify any missing values in the dataset.

In [ ]:
# Check for missing values in each column
display(df.isnull().sum())

### Investigating `long` and `lat` columns
As observed from the descriptive statistics, the `long` and `lat` columns show unusual ranges. Let's inspect these values to understand the extent of the problem and identify potential outliers or incorrectly formatted data.

In [ ]:
print(f"Original 'long' min: {df['long'].min()}, max: {df['long'].max()}")
print(f"Original 'lat' min: {df['lat'].min()}, max: {df['lat'].max()}")

# Define reasonable bounds for latitude and longitude
# Latitude: -90 to 90
# Longitude: -180 to 180

# Count rows where 'long' or 'lat' are outside typical bounds
invalid_long_count = df[(df['long'] < -180) | (df['long'] > 180)].shape[0]
invalid_lat_count = df[(df['lat'] < -90) | (df['lat'] > 90)].shape[0]

print(f"\nNumber of rows with invalid longitude (outside -180 to 180): {invalid_long_count}")
print(f"Number of rows with invalid latitude (outside -90 to 90): {invalid_lat_count}")

# Display some rows with invalid coordinates to understand the pattern
if invalid_long_count > 0 or invalid_lat_count > 0:
    print("\nExamples of rows with invalid coordinates:")
    display(df[(df['long'] < -180) | (df['long'] > 180) | (df['lat'] < -90) | (df['lat'] > 90)].head())

### Cleaning `long` and `lat` columns
Given the anomalous values, I will define a function to validate and clean the `long` and `lat` columns. Values falling outside standard geographical ranges will be replaced with `NaN`.

In [ ]:
import numpy as np

def clean_coordinates(df, long_col='long', lat_col='lat'):
    """
    Cleans longitude and latitude columns by replacing out-of-range values with NaN.
    Assumes standard longitude range [-180, 180] and latitude range [-90, 90].
    """
    df_cleaned = df.copy()

    # Convert columns to numeric, coercing errors to NaN
    df_cleaned[long_col] = pd.to_numeric(df_cleaned[long_col], errors='coerce')
    df_cleaned[lat_col] = pd.to_numeric(df_cleaned[lat_col], errors='coerce')

    # Replace out-of-range longitudes with NaN
    df_cleaned.loc[(df_cleaned[long_col] < -180) | (df_cleaned[long_col] > 180), long_col] = np.nan

    # Replace out-of-range latitudes with NaN
    df_cleaned.loc[(df_cleaned[lat_col] < -90) | (df_cleaned[lat_col] > 90), lat_col] = np.nan

    return df_cleaned

# Apply the cleaning function
df_cleaned = clean_coordinates(df.copy())

# Check the new min/max values and null counts after cleaning
print("\nAfter cleaning:")
print(f"Cleaned 'long' min: {df_cleaned['long'].min()}, max: {df_cleaned['long'].max()}")
print(f"Cleaned 'lat' min: {df_cleaned['lat'].min()}, max: {df_cleaned['lat'].max()}")
print(f"Missing values in 'long' after cleaning: {df_cleaned['long'].isnull().sum()}")
print(f"Missing values in 'lat' after cleaning: {df_cleaned['lat'].isnull().sum()}")

# Display the head of the cleaned DataFrame to see the changes
display(df_cleaned.head())

### Analyzing Categorical Features
Let's examine the unique values and their counts for the categorical columns (`object` dtype) in the `df_cleaned` DataFrame to understand their distribution and identify any potential inconsistencies.

In [ ]:
# Select categorical columns (object dtype)
categorical_cols = df_cleaned.select_dtypes(include='object').columns

print("Unique values and their counts for categorical columns:")
for col in categorical_cols:
    print(f"\n--- Column: {col} ---")
    # Display value counts for top 20 categories, or all if less than 20
    display(df_cleaned[col].value_counts().head(20))

### Analyzing Numerical Features
Next, let's look at the distribution of the numerical features using histograms and box plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select numerical columns (excluding id, and coordinate related columns which we've already cleaned)
numerical_cols = ['tinggi_tower', 'tahun']

plt.figure(figsize=(15, 5))
for i, col in enumerate(numerical_cols):
    plt.subplot(1, 2, i + 1)
    sns.histplot(df_cleaned[col], kde=True)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

plt.figure(figsize=(15, 5))
for i, col in enumerate(numerical_cols):
    plt.subplot(1, 2, i + 1)
    sns.boxplot(y=df_cleaned[col])
    plt.title(f'Box Plot of {col}')
plt.tight_layout()
plt.show()

### Detailed Categorical Feature Analysis
Let's visualize the distributions of some key categorical features, particularly `pemilik_menara` and `struktur_tower`, which might offer insights into ownership patterns and tower types. We'll also check `satuan` for consistency and `keterangan` for its content.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize top N categories for relevant columns
def plot_top_categories(df, column, n=10):
    plt.figure(figsize=(10, 6))
    df[column].value_counts().head(n).plot(kind='barh')
    plt.title(f'Top {n} {column} Distribution')
    plt.xlabel('Count')
    plt.ylabel(column)
    plt.gca().invert_yaxis() # Invert y-axis to have the highest count on top
    plt.show()

# Plot top owners
plot_top_categories(df_cleaned, 'pemilik_menara', n=10)

# Plot structure types
plot_top_categories(df_cleaned, 'struktur_tower', n=5)

# Check unique values for 'satuan' and 'keterangan'
print("\nUnique values for 'satuan':")
display(df_cleaned['satuan'].unique())

print("\nValue counts for 'keterangan':")
display(df_cleaned['keterangan'].value_counts())


### Geospatial Visualization of Tower Locations
With the `long` and `lat` columns cleaned, let's visualize the distribution of these telecommunication towers on an interactive map using `folium`. This will provide a geographical overview of the tower placements.

In [ ]:
import folium

# Filter out rows with NaN in 'long' or 'lat' for mapping
map_df = df_cleaned.dropna(subset=['long', 'lat']).copy()

# Get the average coordinates to center the map (approximated for Bandung)
# Using a known approximate center for Bandung as a default if map_df is empty or coordinates are still off
center_lat = map_df['lat'].mean() if not map_df.empty else -6.9175
center_long = map_df['long'].mean() if not map_df.empty else 107.6191

# Create a base map centered around the average coordinates
m = folium.Map(location=[center_lat, center_long], zoom_start=12)

# Add markers for each tower location
for idx, row in map_df.iterrows():
    if not pd.isna(row['lat']) and not pd.isna(row['long']):
        folium.Marker(
            location=[row['lat'], row['long']],
            popup=f"Owner: {row['pemilik_menara']}<br>Height: {row['tinggi_tower']} {row['satuan']}"
        ).add_to(m)

# Display the map
display(m)

### Google Earth Engine Risk and Coverage Analysis
We will now use GEE to assess the risk of tower sites based on terrain (landslides, slope), precipitation, and population coverage.

**Risk Objective:** Steeper slopes, higher elevations, and greater distances from roads increase mobilization and repair costs.

In [ ]:
import ee
from google.colab import auth

# Authenticate and initialize Earth Engine
auth.authenticate_user()
ee.Initialize(project='rare-cubist-485707-u5')

In [ ]:
# 1. Load Datasets
dem = ee.Image("NASA/NASADEM_HGT/001").select("elevation")
slope = ee.Terrain.slope(dem)
aspect = ee.Terrain.aspect(dem)

# Precipitation (GPM - Global Precipitation Measurement) - V07
precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V07") \
    .filterDate("2020-01-01", "2020-12-31") \
    .select("precipitation") \
    .reduce(ee.Reducer.sum())

# Population (WorldPop 2020)
population = ee.ImageCollection("WorldPop/GP/100m/pop") \
    .filter(ee.Filter.And([
        ee.Filter.eq("year", 2020),
        ee.Filter.eq("country", "IDN")
    ])).first()

# 2. Define Risk Index
def normalize(img):
    min_max = img.reduceRegion(reducer=ee.Reducer.minMax(), scale=1000, bestEffort=True)
    band_name = ee.String(img.bandNames().get(0))
    min_val = ee.Number(min_max.get(band_name.cat("_min")))
    max_val = ee.Number(min_max.get(band_name.cat("_max")))
    return img.subtract(min_val).divide(max_val.subtract(min_val))

slope_norm = normalize(slope)
elevation_norm = normalize(dem)

# Risk Score: Higher slope and higher elevation = higher risk/cost
risk_score = slope_norm.add(elevation_norm).rename("risk_index")

print("GEE Datasets and Risk Model successfully initialized.")

In [ ]:
import folium
from branca.element import Figure

# 3. Sampling Tower Data for GEE Analysis
tower_points = []
# Using a subset of the data to avoid 'Computation timed out'
valid_towers = df_cleaned.dropna(subset=["long", "lat"]).head(500)

for idx, row in valid_towers.iterrows():
    geom = ee.Geometry.Point([row["long"], row["lat"]])
    tower_points.append(ee.Feature(geom, {
        "id_tower": int(row["id"]),
        "owner": str(row["pemilik_menara"])
    }))

tower_fc = ee.FeatureCollection(tower_points)

# Sample the Risk Index, Precipitation, and Population at tower locations
# geometries=True is needed for the map
sampled_data = risk_score.addBands(precip).addBands(population).sampleRegions(
    collection=tower_fc,
    scale=30,
    geometries=True
)

# Export results to Pandas DataFrame
print("Fetching sampled data from GEE (subset of 500 points)...")
results = sampled_data.getInfo()

# Extract properties and coordinates
features_list = []
for f in results['features']:
    props = f['properties']
    props['lon'] = f['geometry']['coordinates'][0]
    props['lat'] = f['geometry']['coordinates'][1]
    features_list.append(props)

risk_df = pd.DataFrame(features_list)

# Rename columns for clarity
risk_df = risk_df.rename(columns={
    "precipitation_sum": "annual_precip_mm",
    "population": "pop_density_2020"
})

# Create Map Visualization
map_center = [risk_df['lat'].mean(), risk_df['lon'].mean()]
m2 = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB positron')

# Define color function for Risk Index
def get_color(risk):
    if risk > 0.8: return 'red'
    if risk > 0.5: return 'orange'
    if risk > 0.3: return 'yellow'
    return 'blue'

# Add circles to the map
for idx, row in risk_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        popup=f"Tower ID: {row['id_tower']}<br>Risk Index: {row['risk_index']:.2f}<br>Owner: {row['owner']}",
        color=get_color(row['risk_index']),
        fill=True,
        fill_opacity=0.7
    ).add_to(m2)

display(m2)
display(risk_df.head())

### Risk Analysis Visualization
Now that we have the sampled data, let's visualize the relationship between the **Risk Index** (terrain difficulty) and the **Population Density** (impact of outage). Towers in the top-right quadrant are critical high-risk/high-impact sites.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 7))
scatter = sns.scatterplot(
    data=risk_df,
    x='risk_index',
    y='pop_density_2020',
    size='annual_precip_mm',
    hue='annual_precip_mm',
    palette='viridis',
    alpha=0.6
)

plt.title('Tower Risk Profile: Terrain Risk vs. Population Impact')
plt.xlabel('Risk Index (Normalized Elevation + Slope)')
plt.ylabel('Population Density (2020)')
plt.legend(title='Annual Precip (mm)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Identify top 10 highest risk towers
high_risk_towers = risk_df.nlargest(10, 'risk_index')
print("Top 10 Towers with Highest Risk Index:")
display(high_risk_towers[['id_tower', 'owner', 'risk_index', 'annual_precip_mm']])

### Investigating Tower ID 418
Let's look at the specific attributes for this tower to understand its high risk score.

In [ ]:
from scipy import stats

# Filter for Tower ID 418 in our sampled risk data
tower_418_risk = risk_df[risk_df['id_tower'] == 418]

# Also get its original cleaning metadata
tower_418_meta = df_cleaned[df_cleaned['id'] == 418]

print("Risk Data for Tower 418:")
display(tower_418_risk)

# Compare to dataset averages
avg_risk = risk_df['risk_index'].mean()
max_risk = risk_df['risk_index'].max()
percentile = stats.percentileofscore(risk_df['risk_index'], 0.823112)

print(f"\nContext:")
print(f"Average Risk Index: {avg_risk:.2f}")
print(f"Maximum Risk Index in subset: {max_risk:.2f}")
print(f"Tower 418 is in the top {100 - percentile:.1f}% highest risk towers.")